In [0]:
-- ---------- FACTS ----------


In [0]:
-- Team performance by week
CREATE OR REPLACE MATERIALIZED VIEW fact_team_week AS
WITH base AS (
  SELECT m.league_id, m.week, m.matchup_id, m.roster_id, m.points AS points_for
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
  -- Filter out future weeks that haven't been played yet
  WHERE m.points > 0
),
opp AS (
  SELECT a.league_id, a.week, a.matchup_id, a.roster_id,
         a.points_for, b.points_for AS points_against
  FROM base a
  LEFT JOIN base b
    ON a.league_id=b.league_id AND a.week=b.week
   AND a.matchup_id=b.matchup_id AND a.roster_id<>b.roster_id
),
li AS (
  SELECT league_id, season FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
wmed AS (
  SELECT o.league_id, l.season, o.week,
         percentile_approx(o.points_for, 0.5) AS med
  FROM opp o JOIN li l USING (league_id)
  GROUP BY o.league_id, l.season, o.week
)
SELECT o.league_id, l.season, o.week, o.roster_id,
       o.points_for, o.points_against,
       (o.points_for >= w.med) AS median_beat_flag
FROM opp o
JOIN li l ON o.league_id = l.league_id
JOIN wmed w ON o.league_id=w.league_id AND l.season=w.season AND o.week=w.week;

In [0]:
-- Player performance by week with position and starter status
CREATE OR REPLACE MATERIALIZED VIEW fact_player_week AS
WITH src AS (
  SELECT m.league_id, li.season, m.week, m.roster_id,
         m.starters, m.points AS total_matchup_points,
         map_entries(m.players_points) AS entries
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
  -- Filter out future weeks that haven't been played yet (0 points = unplayed)
  WHERE m.points > 0
),
rows AS (
  SELECT league_id, season, week, roster_id, starters,
         transform(entries, e -> named_struct('player_id', e.key, 'points', e.value)) AS rows
  FROM src
),
exploded AS (
  SELECT league_id, season, week, roster_id, starters, explode(rows) AS r
  FROM rows
)
SELECT 
  e.league_id, 
  e.season, 
  e.week, 
  e.roster_id,
  e.r.player_id, 
  e.r.points,
  CAST(NULL AS DOUBLE) AS projected_points,
  p.position,
  array_contains(e.starters, e.r.player_id) AS was_started,
  current_timestamp() AS updated_at
FROM exploded e
LEFT JOIN dim_players p ON e.r.player_id = p.player_id;

In [0]:
-- Enriched player week with cluster info
CREATE OR REPLACE MATERIALIZED VIEW fact_player_week_enriched AS
SELECT f.*,
       lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
       li.name AS cluster_name
FROM fact_player_week f
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);


In [0]:
-- Standings by week
CREATE OR REPLACE MATERIALIZED VIEW fact_standings_week AS
WITH wl AS (
  SELECT league_id, season, week, roster_id,
         (points_for > points_against)  AS win_flag,
         (points_for < points_against)  AS loss_flag,
         (points_for = points_against)  AS tie_flag,
         median_beat_flag,
         points_for, points_against
  FROM fact_team_week
)
SELECT league_id, season, week, roster_id,
       SUM(CASE WHEN win_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS wins,
       SUM(CASE WHEN loss_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS losses,
       SUM(CASE WHEN tie_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ties,
       points_for, points_against,
       SUM(CASE WHEN median_beat_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS expected_wins
FROM wl;

In [0]:
-- Waiver acquisitions fact table
CREATE OR REPLACE MATERIALIZED VIEW fact_waiver_acquisitions AS
WITH waiver_adds AS (
  SELECT
    t.transaction_id,
    t.league_id,
    t.leg AS week,
    coalesce(t.waiver_bid, 0) AS faab_bid,
    CASE WHEN t.type = 'waiver' THEN TRUE ELSE FALSE END AS was_waiver,
    to_timestamp(t.created/1000.0) AS acquired_date,
    explode(transform(map_entries(t.adds), e -> 
      named_struct('player_id', e.key, 'to_roster_id', e.value)
    )) AS a
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  WHERE t.type IN ('waiver', 'free_agent')
)
SELECT
  wa.transaction_id,
  wa.league_id,
  li.season,
  wa.week,
  wa.a.to_roster_id AS roster_id,
  wa.a.player_id,
  wa.faab_bid,
  wa.was_waiver,
  wa.acquired_date
FROM waiver_adds wa
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);